# 💸 SpendSmart‑ML — Personalized Spending Recommender (Colab)

Standalone model‑training notebook for a **personalized** (per‑user, not generic) spending
recommendation engine, with **benchmarked accuracy** and a **real‑time** update demo.

**Run order:** Runtime → Run all. Everything is CPU‑only and finishes in a few minutes.

Pipeline: `categorizer → monthly panel → cohorts → personalized forecaster → recommender`.

## 1 · Setup — get the code + install deps

In [ ]:
# Option A: clone from GitHub (edit the URL to your repo), OR
# Option B: upload the `spendsmart-ml` folder via the Files panel and skip the clone.
import os, sys

REPO_URL = ''  # e.g. 'https://github.com/1Akash3/spendsmart-ml.git'
PROJECT_DIR = 'spendsmart-ml'

if REPO_URL and not os.path.isdir(PROJECT_DIR):
    !git clone -q $REPO_URL

# Fallback: if you uploaded files to /content directly, point PROJECT_DIR at them.
if not os.path.isdir(PROJECT_DIR):
    PROJECT_DIR = '.' if os.path.isfile('config.py') else PROJECT_DIR

assert os.path.isdir(PROJECT_DIR), (
    'Upload the spendsmart-ml folder or set REPO_URL. '
    'Expected to find config.py inside PROJECT_DIR.')
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())

In [ ]:
!pip -q install -r requirements.txt
import os, sys; sys.path.insert(0, os.getcwd()); sys.path.insert(0, os.path.join(os.getcwd(),'src'))

# --- Kaggle auth (REAL data is the default). Provide a Kaggle API token: ---
# 1) kaggle.com/settings -> Create New Token  -> downloads kaggle.json
# 2) In Colab, upload it, then:
#    os.environ['KAGGLE_USERNAME'] = '<your_username>'
#    os.environ['KAGGLE_KEY']      = '<your_key>'
# (or run `--synthetic` mode below to skip credentials entirely)

## 2 · Train the whole pipeline

Loads **real** transactions (Kaggle credit-card dataset, ~1.3M rows, 983 users, 18 months),
trains every stage, and back-tests accuracy against naive baselines. Pass `source="synthetic"`
for an offline, no-credentials run.

In [ ]:
from src.train import run
# COMBINED real data by default (credit-card + personal-finance-tracker). Needs a Kaggle token.
metrics = run(source="combined", demo_goal_rate=0.25, seed=42)
# credit-card only:  run(source="real")   |   offline:  run(source="synthetic", users=500, months=18)

## 3 · Proven accuracy (vs naive baselines)

In [ ]:
import json
c, f = metrics['categorizer'], metrics['forecaster']
r = metrics.get('recommender_eval', {})
print(f"Categorizer   : accuracy={c['accuracy']:.3f}  macro-F1={c['macro_f1']:.3f}")
print(f"Forecaster    : MAE={f['model_mae']} vs naive={f['naive_mae']}  "
      f"(skill {f['skill_vs_naive']:+.1%})")
if 'model' in r:
    print(f"Overspend det.: F1={r['model']['f1']} vs naive F1={r['naive_baseline']['f1']}")
print(f"Segmentation  : silhouette={metrics['segmentation']['silhouette']:.3f}")

### Forecaster MAE by category — model vs naive

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
pc = metrics['forecaster']['per_category_mae']
cats = list(pc.keys())
model = [pc[c]['model_mae'] for c in cats]
naive = [pc[c]['naive_mae'] for c in cats]
x = np.arange(len(cats)); w = 0.4
fig, ax = plt.subplots(figsize=(11,4))
ax.bar(x-w/2, model, w, label='Personalized model')
ax.bar(x+w/2, naive, w, label='Naive (last month)')
ax.set_xticks(x); ax.set_xticklabels(cats, rotation=45, ha='right')
ax.set_ylabel('MAE (lower is better)'); ax.set_title('Per-category forecast error')
ax.legend(); plt.tight_layout(); plt.show()

## 4 · A single user's personalized recommendation

Rebuild the artifacts in‑notebook so we can inspect one user end‑to‑end.

In [ ]:
import numpy as np
from data_sources import load_real_transactions
from features import (build_monthly_panel, build_user_profiles, build_forecast_frame,
                      build_serving_frame, PROFILE_FEATURE_COLS)
from segmentation import UserSegmenter
from forecaster import PersonalizedForecaster
from recommender import PersonalizedRecommender, RealTimeState
from config import EXPENSE_CATEGORIES, CATEGORY_LABELS

df = load_real_transactions()              # REAL Kaggle transactions
panel = build_monthly_panel(df)
panel['income'] = panel['income'].fillna(0.0)
profiles = build_user_profiles(panel).replace([np.inf, -np.inf], np.nan).fillna(0.0)
seg = UserSegmenter().fit(profiles)
static = seg.assign(profiles)[PROFILE_FEATURE_COLS + ['cohort']].reset_index()
fc = PersonalizedForecaster().fit(build_forecast_frame(panel, static_features=static))
forecasts = fc.predict_by_user_category(build_serving_frame(panel, static_features=static))
print('Artifacts ready for', len(forecasts), 'real users')

In [ ]:
USER = int(profiles.index[5])   # pick any user id
upanel = panel[panel.user_id == USER]
cohort = int(static.loc[static.user_id == USER, 'cohort'].iloc[0])
out = PersonalizedRecommender().recommend(
    upanel, forecasts[USER], cohort_norms_row=seg.cohort_category_norms_.loc[cohort],
    savings_goal_rate=0.30)

s = out['summary']
print(f"User {USER} | cohort {cohort} | income≈₹{s['avg_income']:,.0f} | "
      f"projected savings rate {s['projected_savings_rate']*100:.1f}%\n")
for i, rec in enumerate(out['recommendations'], 1):
    print(f"{i}. [{rec['kind']}] {rec['title']}")
    print(f"   {rec['detail']}")
    print(f"   ~₹{rec['monthly_impact']:,.0f}/mo  (confidence {rec['confidence']:.0%})\n")
if out['plan']:
    print('Savings plan:', out['plan']['summary'])

### That user's spending history + next‑month forecast

In [ ]:
import matplotlib.pyplot as plt
g = upanel.sort_values('month')
top = g[EXPENSE_CATEGORIES].mean().sort_values(ascending=False).head(5).index.tolist()
fig, ax = plt.subplots(figsize=(11,4))
for c in top:
    ax.plot(g['month'], g[c], marker='o', label=CATEGORY_LABELS[c])
    ax.scatter([g['month'].max()], [forecasts[USER][c]], marker='*', s=180, zorder=5)
ax.set_title(f'User {USER}: top categories (★ = next-month forecast)')
ax.set_ylabel('₹ / month'); ax.legend(); plt.tight_layout(); plt.show()

## 5 · Real‑time optimization demo

Feed transactions as they 'arrive' mid‑month; the online layer projects the
month‑end total and fires an alert in O(#categories) — no retrain.

In [ ]:
mean = {c: float(upanel[c].mean()) for c in EXPENSE_CATEGORIES}
std  = {c: float(upanel[c].std(ddof=0)) for c in EXPENSE_CATEGORIES}
rt = RealTimeState(mean, std, income=float(upanel['income'].mean()))

# Simulate a heavy dining week early in the month.
stream = [('food_dining', mean['food_dining']*0.5, 3),
          ('food_dining', mean['food_dining']*0.6, 6),
          ('shopping',    mean['shopping']*0.4,   7),
          ('food_dining', mean['food_dining']*0.7, 9)]
for cat, amt, day in stream:
    for a in rt.update(cat, amt, day):
        print(f"day {day:>2}  ⚠  {a['message']}")
print('\nProjected month-end (top 5):')
pe = rt.projected_month_end()
for c in sorted(pe, key=pe.get, reverse=True)[:5]:
    print(f"  {CATEGORY_LABELS[c]:<16} ₹{pe[c]:,.0f}")

## 6 · (Optional) Train on REAL public datasets

Pull the 4.5M‑row HuggingFace categorization set (MIT) or Kaggle sets. Needs
`datasets` / `kagglehub` + credentials. See `data/README.md` for the full catalog.

In [ ]:
# !pip -q install datasets kagglehub
# Our registry lives in src/data_sources.py (named to NOT clash with HF's `datasets`).
from data_sources import REGISTRY, print_catalog
print_catalog()

# Example (uncomment after installing `datasets` + `huggingface-cli login`):
# from data_sources import load_hf_transaction_categorization
# real = load_hf_transaction_categorization(sample=100_000)   # 4.5M-row MIT set, sampled
# from categorizer import TransactionCategorizer
# from sklearn.model_selection import train_test_split
# Xtr,Xte,ytr,yte = train_test_split(real['description'], real['category'], test_size=0.2)
# print(TransactionCategorizer().fit(Xtr,ytr).evaluate(Xte,yte))

---
Everything here is decoupled from the SpendSmart web app — this notebook only
trains and evaluates the model. Integration is a separate, later step.